# Day 19: Containers and Azure Deployment
**Containerising apps, Azure App Service, Azure Container Apps and continuous deployment**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.

## 1. Setup
We make the app cloud ready and test it here, then generate the Docker, compose and pipeline files. Azure commands run in **Azure Cloud Shell** (no install) or a VS Code terminal with the Azure CLI.

In [ ]:
# ===== Option A: Google Colab (active) =====
!pip install -q fastapi httpx pyyaml

# ===== Option B: VS Code / local =====
# pip install fastapi uvicorn httpx pyyaml
import os, shutil
shutil.rmtree("deploy-demo", ignore_errors=True)
os.makedirs("deploy-demo/app", exist_ok=True)
os.chdir("deploy-demo")
print("Working in", os.getcwd())

## 2. A cloud-ready API

In [ ]:
%%writefile app/main.py
import logging, os, sys
from fastapi import FastAPI
from fastapi.responses import JSONResponse

logging.basicConfig(stream=sys.stdout, level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger("tasks-api")

APP_ENV = os.getenv("APP_ENV", "local")
APP_VERSION = os.getenv("APP_VERSION", "dev")
DATABASE_URL = os.getenv("DATABASE_URL", "sqlite:///local.db")

app = FastAPI(title="Tasks API")

def database_reachable() -> bool:
    # Real apps ping the database here. We simulate with an env flag for the demo.
    return os.getenv("SIMULATE_DB_DOWN") != "1"

@app.get("/health")
def health():
    return {"status": "ok", "env": APP_ENV, "version": APP_VERSION}

@app.get("/ready")
def ready():
    if not database_reachable():
        log.warning("Readiness failed: database unreachable")
        return JSONResponse({"ready": False}, status_code=503)
    return {"ready": True}

In [ ]:
import importlib, sys
from fastapi.testclient import TestClient

def fresh_app(**env):
    for k in ["APP_ENV", "APP_VERSION", "SIMULATE_DB_DOWN"]:
        os.environ.pop(k, None)
    os.environ.update(env)
    sys.modules.pop("app.main", None)
    import app.main as m
    return TestClient(m.app)

print("Local:", fresh_app().get("/health").json())
print("Prod: ", fresh_app(APP_ENV="production", APP_VERSION="1.4.2").get("/health").json())
c = fresh_app(SIMULATE_DB_DOWN="1")
print("Ready with DB down:", c.get("/ready").status_code, c.get("/ready").json())

### TODO 1: Add a `CORS_ORIGINS` environment variable (comma separated) and use it to configure `CORSMiddleware`.

In [ ]:
# TODO: write your code here

## 3. Container files

In [ ]:
%%writefile requirements.txt
fastapi
uvicorn

In [ ]:
%%writefile Dockerfile
FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app/ ./app
RUN useradd -m appuser
USER appuser
ENV APP_ENV=container
EXPOSE 8000
HEALTHCHECK CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:8000/health')"
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]

In [ ]:
%%writefile docker-compose.yml
services:
  api:
    build: .
    ports:
      - "8000:8000"
    environment:
      APP_ENV: local-compose
      APP_VERSION: compose
      MONGO_URI: mongodb://db:27017
    depends_on:
      - db
  db:
    image: mongo:7
    volumes:
      - mongo-data:/data/db
volumes:
  mongo-data:

In [ ]:
import yaml
compose = yaml.safe_load(open("docker-compose.yml"))
for name, svc in compose["services"].items():
    print(f"{name:<4} image={svc.get('image', 'built from Dockerfile'):<22} ports={svc.get('ports', [])}")

# ===== Option B: VS Code with Docker Desktop =====
# docker compose up --build
# curl http://localhost:8000/health
# docker compose down

## 4. Deploy with the Azure CLI (Cloud Shell)
Upload this folder to Cloud Shell (or clone your repo there). Names must be globally unique, so add your initials and a number. **Delete the resource group when you finish.**

In [ ]:
# ===== Option A: Azure Cloud Shell (Bash) =====
# RG=rg-day19-yourname; LOC=centralindia; ACR=acrday19yourname
# az group create -n $RG -l $LOC --tags owner=yourname project=training
# az acr create -g $RG -n $ACR --sku Basic
# az acr build -r $ACR -t tasks-api:1.0 .
#
# --- Option 1: Azure Container Apps (simplest) ---
# az containerapp up --name tasks-api --resource-group $RG --location $LOC \
#   --source . --ingress external --target-port 8000 --env-vars APP_ENV=dev APP_VERSION=1.0
#
# --- Option 2: Azure App Service for containers ---
# az appservice plan create -g $RG -n plan-day19 --is-linux --sku B1
# az webapp create -g $RG -p plan-day19 -n tasks-api-yourname --container-image-name $ACR.azurecr.io/tasks-api:1.0
# az webapp config appsettings set -g $RG -n tasks-api-yourname --settings WEBSITES_PORT=8000 APP_ENV=dev
#
# --- Clean up ---
# az group delete -n $RG --yes --no-wait

# ===== Option B: VS Code terminal =====
# Install the Azure CLI, run az login, then use the same commands.
print("Azure commands are listed above")

## 5. The CD pipeline

In [ ]:
%%writefile azure-pipelines.yml
trigger:
  - main

variables:
  acrServiceConnection: sc-acr          # Docker registry service connection
  azureSubscription: sc-azure            # Azure Resource Manager service connection
  repository: tasks-api
  acrLoginServer: acrday19yourname.azurecr.io
  tag: $(Build.BuildId)

stages:
  - stage: Test
    jobs:
      - job: unit
        pool: { vmImage: ubuntu-latest }
        steps:
          - task: UsePythonVersion@0
            inputs: { versionSpec: "3.11" }
          - script: pip install -r requirements.txt pytest httpx && python -m pytest -q || echo "add tests"
            displayName: Tests

  - stage: Build
    dependsOn: Test
    jobs:
      - job: image
        pool: { vmImage: ubuntu-latest }
        steps:
          - task: Docker@2
            displayName: Build and push image
            inputs:
              command: buildAndPush
              containerRegistry: $(acrServiceConnection)
              repository: $(repository)
              tags: $(tag)

  - stage: DeployDev
    dependsOn: Build
    jobs:
      - deployment: dev
        environment: dev
        pool: { vmImage: ubuntu-latest }
        strategy:
          runOnce:
            deploy:
              steps:
                - task: AzureContainerApps@1
                  inputs:
                    azureSubscription: $(azureSubscription)
                    containerAppName: tasks-api-dev
                    resourceGroup: rg-day19-dev
                    imageToDeploy: $(acrLoginServer)/$(repository):$(tag)

  - stage: DeployProd
    dependsOn: DeployDev
    jobs:
      - deployment: prod
        environment: production          # add an approval check on this environment
        pool: { vmImage: ubuntu-latest }
        strategy:
          runOnce:
            deploy:
              steps:
                - task: AzureContainerApps@1
                  inputs:
                    azureSubscription: $(azureSubscription)
                    containerAppName: tasks-api
                    resourceGroup: rg-day19-prod
                    imageToDeploy: $(acrLoginServer)/$(repository):$(tag)

In [ ]:
p = yaml.safe_load(open("azure-pipelines.yml"))
for st in p["stages"]:
    job = st["jobs"][0]
    env = job.get("environment", "")
    print(f"{st['stage']:<11} after {st.get('dependsOn', '-'):<10} {'environment=' + env if env else ''}")

### TODO 2: Add a `SmokeTest` step after the prod deployment that calls `/health` with `curl --fail --retry 5`.

In [ ]:
# TODO: write your code here

---
**Ganbatte** (頑張って): *Do your best*

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)